# **Schema**
- Nuts and Bolts of working with Large Language Models (LLMs)

**Problem Statement**
- Design and implement a schema for interacting with Large Language Models (LLMs) that ensures consistent, structured, and efficient communication.

## **Text**
- The natural language way to interact with LLMs

In [1]:
%%capture
# update or install the necessary libraries
%pip install --upgrade langchain_core langchain_community langchain_aws python-dotenv

In [2]:
%pip show langchain_core

Name: langchain-core
Version: 1.5.2
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /Users/vijay/Desktop/Ness/workspace/nees_langchain/.myenv/lib/python3.13/site-packages
Requires: jsonpatch, langchain-protocol, langsmith, packaging, pydantic, pyyaml, tenacity, typing-extensions, uuid-utils
Required-by: langchain-aws, langchain-classic, langchain-community, langchain-text-splitters
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from langchain_aws import ChatBedrock
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"] = os.getenv("AWS_DEFAULT_REGION")

In [7]:
# You'll be working with simple strings as prompts(that'll soon grow in complexity!)
my_text = "what all education streams possible after 10th in india, wrap the response in html elments with ordered list and dont use new line characters in html content"
my_text

'what all education streams possible after 10th in india, wrap the response in html elments with ordered list and dont use new line characters in html content'

In [8]:
from langchain_core.messages import HumanMessage
chat = ChatBedrock(
    model_id="mistral.mistral-7b-instruct-v0:2",
    temperature=0.7,
)
response = chat.invoke([HumanMessage(content=my_text)])
response.content

Error raised by bedrock service
Traceback (most recent call last):
  File "/Users/vijay/Desktop/Ness/workspace/nees_langchain/.myenv/lib/python3.13/site-packages/langchain_aws/llms/bedrock.py", line 1173, in _prepare_input_and_invoke
    response = self.client.invoke_model(**request_options)
  File "/Users/vijay/Desktop/Ness/workspace/nees_langchain/.myenv/lib/python3.13/site-packages/botocore/client.py", line 606, in _api_call
    return self._make_api_call(operation_name, kwargs)
           ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vijay/Desktop/Ness/workspace/nees_langchain/.myenv/lib/python3.13/site-packages/botocore/context.py", line 123, in wrapper
    return func(*args, **kwargs)
  File "/Users/vijay/Desktop/Ness/workspace/nees_langchain/.myenv/lib/python3.13/site-packages/botocore/client.py", line 1094, in _make_api_call
    raise error_class(parsed_response, operation_name)
botocore.exceptions.ClientError: An error occurred (UnrecognizedClientException) when c

ClientError: An error occurred (UnrecognizedClientException) when calling the InvokeModel operation: The security token included in the request is invalid.

In [5]:
response

AIMessage(content=' The day that comes after Friday is Saturday. In terms of numbers, if we consider Sunday as the first day of the week, then Friday is the fifth day and Saturday is the sixth day.', additional_kwargs={'usage': {'prompt_tokens': 14, 'completion_tokens': 40, 'cache_read_input_tokens': 0, 'cache_write_input_tokens': 0, 'total_tokens': 54}, 'stop_reason': None, 'thinking': {}, 'model_id': 'mistral.mistral-7b-instruct-v0:2'}, response_metadata={'usage': {'prompt_tokens': 14, 'completion_tokens': 40, 'cache_read_input_tokens': 0, 'cache_write_input_tokens': 0, 'total_tokens': 54}, 'stop_reason': None, 'thinking': {}, 'model_id': 'mistral.mistral-7b-instruct-v0:2', 'model_provider': 'bedrock', 'model_name': 'mistral.mistral-7b-instruct-v0:2'}, id='lc_run--61edba29-3a25-4823-ba26-747963190aee-0', usage_metadata={'input_tokens': 14, 'output_tokens': 40, 'total_tokens': 54, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}})

## **Chat Messages**
Like text, but specified with a message type (System, Human, AI)

* **System** - Helpful background context that tell the AI what to do
* **Human** - Messages that are intented to represent the user
* **AI** - Messages that show what the AI responded with

In [16]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

Now let's create a few messages that simulate a chat experience with a bot


In [17]:
# Chat messages example
response = chat.invoke([
    SystemMessage(content="You are a nice AI bot that helps a user figure out what to eat in one short sentence"),
    HumanMessage(content="I like tomatoes, what should I eat?")
])
print(response.content)

 Consider making a fresh tomato salad or a caprese appetizer for a delicious and satisfying meal.



You can also pass more chat history w/ responses from the AI

In [ ]:
response = chat.invoke(
    [
        SystemMessage(content="You are a nice AI bot that helps a user figure out where to travel in one short sentence"),
        HumanMessage(content="I like the beaches where should I go?"),
        AIMessage(content="You should go to Nice, France"),
        HumanMessage(content="What else should I do when I'm there?")
    ]
)
response.content

You can also exclude the system message if you want

In [ ]:
response = chat.invoke(
    [
        HumanMessage(content="What day comes after Thursday?")
    ]
)
response.content

## **Documents**
An object that holds a piece of text and metadata (more information about that text)

In [18]:
from langchain_core.documents import Document

In [19]:
document = Document(page_content="This is my document. It is full of text that I've gathered from other places",
         metadata={
             'my_document_id' : 234234,
             'my_document_source' : "The LangChain Papers",
             'my_document_create_time' : 1680013019
         })

In [20]:
# Prepare the prompt by combining the document content with your custom prompt
prompt = f"Document Content: {document.page_content}\n\nPlease summarize the above document."

In [21]:
prompt

"Document Content: This is my document. It is full of text that I've gathered from other places\n\nPlease summarize the above document."

In [22]:
# Get the response from the language model
response = chat.invoke([HumanMessage(content=f"Summarize the following document:\n\n{prompt}")])
response.content

' The document is a personal document labeled as "my document." Its content consists of text that the document owner has collected from various sources. There is no specific information or topic discussed within the text.'

But you don't have to include metadata if you don't want to

In [ ]:
document1 = Document(page_content="This is my document. It is full of text that I've gathered from other places")

In [ ]:
# Prepare the prompt by combining the document content with your custom prompt
prompt1 = f"Document Content: {document1.page_content}\n\nPlease summarize the above document."

In [ ]:
# Get the response from the language model
response = chat.invoke([HumanMessage(content=f"Summarize the following document:\n\n{prompt1}")])
response.content

# **Let's Do an Activity**

Create a simple interaction using LangChain and AWS. Define system, human, and AI messages to simulate a conversation with a travel recommendation bot. Additionally, create a document schema to store information about travel destinations.

**Steps**

* **Set Up**: Install the necessary libraries and set up your AWS Secret Key and Access Key.
* **Define Messages**: Create system, human, and AI messages for a travel bot.
* **Simulate Interaction**: Use the chat function to simulate a conversation.
* **`Create Document`**: Define a document schema to store information about a travel destination.